# Function Calling
- Function Calling은 OpenAI API에서 **특정 함수 정의(JSON schema 기반)** 를 주고, 모델이 해당 **함수를 호출하는 형식의 응답** 을 생성하게 하는 기능입니다.
- 챗봇, 에이전트(Agent), 도구(Tool) 사용 등 다양한 자동화 및 플러그인 연동에 활용됩니다.

- 지원 모델 gpt-3-turbo, gpt-4, gpt-4-turbo  (2025.1 현재)

- 모델이 우리의 코드를 호출하도록 해서 우리의 함수들을 모델이 호출할수 있도록 하거나
- 모델이 우리가 원하는 특정 모양과 형식의 output 을 갖도록 강제할수 있다.

- https://platform.openai.com/docs/guides/function-calling?api-mode=chat

![](https://cdn.thenewstack.io/media/2024/05/5026cffa-rag_function_call-1024x835.jpeg)

출처: https://thenewstack.io/a-comprehensive-guide-to-function-calling-in-llms/?utm_source=chatgpt.com


In [1]:
import os 

from dotenv import load_dotenv
load_dotenv()

print(f'OPENAI_API_KEY={os.getenv("OPENAI_API_KEY")[:20]}...')

OPENAI_API_KEY=sk-proj-iKU13YeoxNgF...


In [2]:
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
)

prompt = PromptTemplate.from_template("Who is the weather in {city}")
chain = prompt | llm
response = chain.invoke({'city': "Rome"})
print(response.content)




I can't provide real-time weather updates. To get the current weather in Rome, I recommend checking a reliable weather website or app for the latest information.


In [4]:
# 위치만 제공해주면, 실시간 날씨 정보를 가져오는 함수가 있다고 하자.

import httpx
def get_weather(lon, lat):  # <- 위치 정보 (경도, 위도)
    print(f'🔵 get_weather({lon}, {lat}) 호출')
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    response = httpx.get(url)
    response.raise_for_status()   # http 통신 응답 에러시 예외 발생
    return response.json()


In [6]:
coords_seoul = {
    "lon": 126.9780,
    "lat": 37.5665,
}
get_weather(**coords_seoul)

🔵 get_weather(126.978, 37.5665) 호출


{'latitude': 37.55,
 'longitude': 127.0,
 'generationtime_ms': 0.10001659393310547,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 34.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-11-12T10:30',
  'interval': 900,
  'temperature': 11.8,
  'windspeed': 4.1,
  'winddirection': 232,
  'is_day': 0,
  'weathercode': 1}}

### JSON schema 란?

- https://json-schema.org/overview/what-is-jsonschema


## 함수의 스키마

In [10]:
# 함수의 작동방식과 필요한 것을 설명해 주는 것이 함수의 스키마다

function_schema = {
    "name": "get_weather",   # 함수의 이름 
    # description : 함수가 무슨 일을 하는지 기술
    # ↓'위도와 경도를 받아서 특정장소의 날씨정보를 가져오는 함수"
    "description": "function that takes longitude and latitude to find the weather of a place", 

    # 파라미터 기술
    "parameters": {
        "type": "object",
        "properties": {
            # 경도
            "lon": {"type": "string", "description": "The longitude coordinate"},
            # 위도
            "lat": {"type": "string", "description": "The latitude coordinate"},
        },
    },

    # 필수 사항 기술
    "required": ["lon", "lat"],  # 경도와 위도 속성이 모두 필요하다고 알려준다.
    
}


### ChatOpenAI 에 함수 전달하기 

In [11]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
).bind(    # <- ChatOpenAI 에게 전달할 인수들을 추가
    functions=[    # <- 원하는 만큼 많은 함수를 전달할수 있다.
        function_schema,  # 위에서 준비한 함수 schema 전달
    ],

    # ↓기본적으로 모델이 특정 함수를 사용하도록 '강제'하거나, 
    #   모델이 함수를 사용하도록 하거나 그냥 답변을 할 수 있도록 모델 스스로 선택하게 할수 있다.

    # 모델을 강제로 함수를 사용하도록 하려면! 아래와 같이 해야 한다
    function_call = {
        "name": "get_weather",    # <-- get_weather 함수를 반드시 사용하라!
    },
)

In [13]:
chain = prompt | llm
response = chain.invoke({"city": "Rome"})
print(response.content)  # content 내용이 없다! 으잉?

response

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"lon":"12.4964","lat":"41.9028"}', 'name': 'get_weather'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 79, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e327275518', 'id': 'chatcmpl-EjMrPv4Bypr1lA3ljMDYQHGyXPJib', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1cf6c-3b28-7890-90e7-002f48bb0e13-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 15, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'o

In [ ]:
"""
↑ 함 들여다보자.

AIMessage(
    content='',   # <-- 응답은 받았지만 content 는 없다.  대신! 함수가 호출된것을 볼수 있다!
    additional_kwargs={
        # ↓ 함수가 호출되었다!  
        # 모델은 get_weather 라는 함수를 호출해 줬으면 해!  라고 말하고 있는거다.
        # 그리고, "네가 경도와 위도에 대한 argument 를 제공해 줬으면 해" 라고 하고 있는 거다.        
        # Rome 의 lon 과 lat 값을 전달해 주었다!  오~!
        'function_call': {'arguments': '{"lon":"12.4964","lat":"41.9028"}', 'name': 'get_weather'}, 
        'refusal': None
    }, 
    ...
)
"""
None

In [14]:
response.additional_kwargs

{'function_call': {'arguments': '{"lon":"12.4964","lat":"41.9028"}',
  'name': 'get_weather'},
 'refusal': None}

In [15]:
response.additional_kwargs['function_call']

{'arguments': '{"lon":"12.4964","lat":"41.9028"}', 'name': 'get_weather'}

In [17]:
json_str = response.additional_kwargs['function_call']['arguments']
json_str

'{"lon":"12.4964","lat":"41.9028"}'

In [18]:
import json
r = json.loads(json_str)
r

{'lon': '12.4964', 'lat': '41.9028'}

In [19]:
r['lon'], r['lat']

('12.4964', '41.9028')

In [21]:
get_weather(r['lon'], r['lat'])

🔵 get_weather(12.4964, 41.9028) 호출


{'latitude': 41.895107,
 'longitude': 12.526347,
 'generationtime_ms': 0.06210803985595703,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 58.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-11-12T11:00',
  'interval': 900,
  'temperature': 20.7,
  'windspeed': 24.3,
  'winddirection': 153,
  'is_day': 1,
  'weathercode': 61}}

In [22]:
get_weather(**r)

🔵 get_weather(12.4964, 41.9028) 호출


{'latitude': 41.895107,
 'longitude': 12.526347,
 'generationtime_ms': 0.09334087371826172,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 58.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-11-12T11:00',
  'interval': 900,
  'temperature': 20.7,
  'windspeed': 24.3,
  'winddirection': 153,
  'is_day': 1,
  'weathercode': 61}}

# 퀴즈 생성하기 스키마

In [23]:
function_schema = {
    "name": "create_quiz",   # 실제로는 존재하지 않는 함수다.
    "description": "function that takes a list of questions and answers and returns a quiz",  

    "parameters": {
# ↓ 우라기 원하는 응답의 스키마는
 
# 바로 이 형태를 정의하는 거다. 
# {
#   "questions":[
#      0:{
#        "question":"What … in the story?"
#        "answers":[
#           0:{
#             "answer":"John"
#             "correct":false
#           }
#           ... 
#        ]
#      }
#      ...
# }         
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "items": {   # items <- 배열의 원소 스키마
                    "type": "object",
                    "properties": {
                        "question": {"type": "string"},
                        "answers": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "answer": {"type": "string"},
                                    "correct": {"type": "boolean"},
                                },
                                "required": ["answer", "correct"],
                            },                            
                        },
                    }, 
                    "required": ["question", "answers"],
                },                
            },
        },
        "required": ["questions"],
    },
    
}

In [24]:
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.1,
).bind(
    functions=[function_schema],

    # function_call="auto"  # "auto" 모델이 우리 함수를 사용할수도 있고 사용하지 않을수도 있다.
    function_call={"name": "create_quiz"},
)

prompt = PromptTemplate.from_template("Make a quiz about {city}")
chain = prompt | llm
response = chain.invoke({"city": "Rome"})

response = response.additional_kwargs['function_call']['arguments']

json.loads(response)

{'questions': [{'question': 'What is the capital city of Italy?',
   'answers': [{'answer': 'Rome', 'correct': True},
    {'answer': 'Milan', 'correct': False},
    {'answer': 'Naples', 'correct': False},
    {'answer': 'Florence', 'correct': False}]},
  {'question': 'Which ancient structure is known as the largest amphitheater ever built?',
   'answers': [{'answer': 'The Colosseum', 'correct': True},
    {'answer': 'The Pantheon', 'correct': False},
    {'answer': 'The Roman Forum', 'correct': False},
    {'answer': 'The Circus Maximus', 'correct': False}]},
  {'question': 'What river runs through the city of Rome?',
   'answers': [{'answer': 'Tiber', 'correct': True},
    {'answer': 'Po', 'correct': False},
    {'answer': 'Arno', 'correct': False},
    {'answer': 'Adige', 'correct': False}]},
  {'question': 'Which famous Roman general crossed the Rubicon River, leading to a civil war?',
   'answers': [{'answer': 'Julius Caesar', 'correct': True},
    {'answer': 'Augustus', 'correct':

In [ ]:
"""
↑ 이것이 agent 의 기본이다
- 도구를 만들 때, 그 도구를 형식화(포맷, 스키마) 하고 
- LLM 이 FunctionCalling 으로 응답하고 
- 실제로 컴퓨터에서 이를 함수 호출하는 거다


함수를 어떻게 만들었느냐에 따라 위험할수도 있고, 그렇지 않을수도 있다..
  
"""
None

# Structured Output

**.with_structured_output** 을 적용하면 LLM이 지정한 형식의 JSON(Function Call)으로만 응답합니다.

In [25]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [26]:
# QuizGPT 의 JSON 포맷 구조와 동일한 Pydantic 구조 정의

# 1. 보기 1개에 대한 설계
class AnswerItem(BaseModel):
    answer: str = Field(description="보기에 해당하는 텍스트 내용")
    correct: bool = Field(description="이 보기가 정답이면 true, 오답이면 false")

# 2. 문제 1개 (질문 + 보기4개)에 대한 설계
class QuestionItem(BaseModel):
    question: str = Field(description="시험 문제 내용")
    answers: List[AnswerItem] = Field(description="4개의 보기 목록 (정답 1개, 오답 3개)")

# 3. 전체 응답 (문제 10개 묶음) 에 대한 설계
class QuizResponse(BaseModel):
    questions: List[QuestionItem] = Field(description="생성된 10개의 시험 문제 리스트")


In [27]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# .with_structured_output을 적용하면 LLM이 지정한 형식의 JSON(Function Call)으로만 응답합니다.
structured_llm = llm.with_structured_output(QuizResponse)

# ↑ 이 한 줄이 핵심입니다. LangChain이 내부적으로 다음을 처리합니다.

# - QuizResponse를 JSON Schema로 변환
# - 그 스키마를 OpenAI의 function calling(또는 response_format) 형태로 API 요청에 첨부
# - 모델이 반환한 JSON을 파싱하고 Pydantic 객체로 검증·변환





In [28]:
quiz_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
    You are a helpful assistant that is role playing as a teacher.
         
    Based ONLY on the following context, make 2 questions to test the user's knowledge about the text.
   
    Each question should have 4 answers, three of them must be incorrect and one should be correct.
         
    Context: {context}
    """,
        ),
    ]
)

In [30]:
chain = {"context": lambda x: x} | quiz_prompt | structured_llm
response = chain.invoke("Rome")  # chain 호출의 리턴값은 QuizResponse 다!

type(response)

{"questions":[{"question":"What was the primary language spoken in ancient Rome?","answers":[{"answer":"Latin","correct":true},{"answer":"Greek","correct":false},{"answer":"Hebrew","correct":false},{"answer":"Arabic","correct":false}]},{"question":"Which of the following was a significant architectural achievement of ancient Rome?","answers":[{"answer":"The Colosseum","correct":true},{"answer":"The Great Wall of China","correct":false},{"answer":"The Pyramids of Giza","correct":false},{"answer":"The Parthenon","correct":false}]}]}

D:\MCP2607\LangWork\.venv\Lib\site-packages\pydantic\main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QuizResponse(questions=[Q...non', correct=False)])]), input_type=QuizResponse])
  return self.__pydantic_serializer__.to_python(


__main__.QuizResponse

In [31]:
print(response.model_dump_json(indent=2))

{
  "questions": [
    {
      "question": "What was the primary language spoken in ancient Rome?",
      "answers": [
        {
          "answer": "Latin",
          "correct": true
        },
        {
          "answer": "Greek",
          "correct": false
        },
        {
          "answer": "Hebrew",
          "correct": false
        },
        {
          "answer": "Arabic",
          "correct": false
        }
      ]
    },
    {
      "question": "Which of the following was a significant architectural achievement of ancient Rome?",
      "answers": [
        {
          "answer": "The Colosseum",
          "correct": true
        },
        {
          "answer": "The Great Wall of China",
          "correct": false
        },
        {
          "answer": "The Pyramids of Giza",
          "correct": false
        },
        {
          "answer": "The Parthenon",
          "correct": false
        }
      ]
    }
  ]
}


In [32]:
# Pydantic -> dict 변환
response.model_dump()

{'questions': [{'question': 'What was the primary language spoken in ancient Rome?',
   'answers': [{'answer': 'Latin', 'correct': True},
    {'answer': 'Greek', 'correct': False},
    {'answer': 'Hebrew', 'correct': False},
    {'answer': 'Arabic', 'correct': False}]},
  {'question': 'Which of the following was a significant architectural achievement of ancient Rome?',
   'answers': [{'answer': 'The Colosseum', 'correct': True},
    {'answer': 'The Great Wall of China', 'correct': False},
    {'answer': 'The Pyramids of Giza', 'correct': False},
    {'answer': 'The Parthenon', 'correct': False}]}]}